# Rainfall & Observed vs. Simulated Discharge — Annual Panel Plot

**Purpose:** Produces a one-panel annual plot showing daily observed and
simulated discharge on the left axis, and precipitation as an inverted bar
chart on the right axis, laid out month-by-month.

**What it does:**
- Reads observed discharge and simulated discharge from file
- Reads precipitation time series
- Plots a combined hyetograph–hydrograph for a single chosen year
- Optionally fills the area under the simulated discharge curve

**Input:** Observed Q Excel, simulated Q CSV, precipitation CSV  
**Output:** Annual hyetograph–hydrograph PNG

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import MonthLocator, DateFormatter

def plot_year_onepanel(year: int, save_path: str | None = None,
                       bar_width_days: float = 1.1,
                       fill_discharge: str | None = None):
    """
    One-panel plot, months on x-axis:
      - Left y-axis: Discharge (Obs blue, Sim red) [daily mean]
      - Right y-axis: Rainfall bars (green) [daily sum], axis inverted (bars from top)
      - X-limits forced to full calendar year without margins
      - Rainfall reindexed to full year (missing days -> 0)
      - Discharge reindexed to full year (missing days -> NaN by default, or fill via 'ffill'/'bfill'/'interpolate')

    Args:
        year: e.g., 2014
        save_path: optional filepath to save the figure
        bar_width_days: width of daily bars (in "days" units). Try 1.0–1.2 for chunkier look.
        fill_discharge: None (default) keeps NaN; set to 'ffill', 'bfill', or 'interpolate' to fill daily gaps.
    """
    # Full calendar span for the year
    year_start = pd.Timestamp(year=year, month=1, day=1)
    year_end   = pd.Timestamp(year=year, month=12, day=31, hour=23, minute=59, second=59)

    # Filter original data (df_flow, df_rain must exist in your session as before)
    flow_y = df_flow[(df_flow["Zeit"] >= year_start) & (df_flow["Zeit"] <= year_end)].copy()
    rain_y = df_rain[(df_rain["Zeit"] >= year_start) & (df_rain["Zeit"] <= year_end)].copy()

    if flow_y.empty and rain_y.empty:
        print(f"⚠️ No rainfall or discharge data found for {year}.")
        return

    # ---- Daily aggregation ----
    # Rainfall: sum per day
    if not rain_y.empty:
        rain_day = (rain_y.set_index("Zeit")["precip"]
                    .resample("D").sum())
    else:
        rain_day = pd.Series(dtype=float)

    # Discharge: mean per day
    if not flow_y.empty:
        flow_day = (flow_y.set_index("Zeit")[["Obs", "Sim"]]
                    .resample("D").mean())
    else:
        flow_day = pd.DataFrame(columns=["Obs", "Sim"])

    # ---- Reindex to a full daily calendar ----
    full_days = pd.date_range(year_start.normalize(), year_end.normalize(), freq="D")

    # Rainfall: missing days -> 0 (no rain)
    rain_day = rain_day.reindex(full_days).fillna(0.0)

    # Discharge: missing days left as NaN by default (no artificial data)
    flow_day = flow_day.reindex(full_days)
    if fill_discharge == "ffill":
        flow_day = flow_day.ffill()
    elif fill_discharge == "bfill":
        flow_day = flow_day.bfill()
    elif fill_discharge == "interpolate":
        flow_day = flow_day.interpolate(limit_direction="both")

    # ---- Prepare for plotting ----
    # Move index to columns for plotting
    rain_df = rain_day.reset_index()
    rain_df.columns = ["Zeit", "precip"]

    flow_df = flow_day.reset_index().rename(columns={"index": "Zeit"})

    # ---- Figure (one panel + twin y-axis) ----
    fig, ax_left = plt.subplots(1, 1, figsize=(16, 8))  # slightly taller if you prefer: 15 x 5.5
    ax_right = ax_left.twinx()

    # Discharge (left axis) — plot only if columns exist
    if not flow_df.empty and {"Obs", "Sim"}.issubset(flow_df.columns):
        ax_left.plot(flow_df["Zeit"], flow_df["Obs"], color="blue", linewidth=2, label="Observed")
        ax_left.plot(flow_df["Zeit"], flow_df["Sim"], color="red",  linewidth=2, label="Simulated")
    ax_left.set_ylabel("Discharge (m³/s)")
    ax_left.grid(True, alpha=0.3)

    # Rainfall (right axis) as bars from the top
    if not rain_df.empty:
        ax_right.bar(rain_df["Zeit"], rain_df["precip"],
                     width=bar_width_days, color="green", alpha=0.7, edgecolor="none")
        ax_right.invert_yaxis()  # bars hang from the top
        ax_right.set_ylabel("Daily Rainfall (mm)", color="green")
        ax_right.tick_params(axis="y", colors="green")

    # Title
    ax_left.set_title(f"Daily Rainfall–Discharge Response ({year})", fontweight="bold")

    # X-axis: months, no margins, exact year limits
    ax_left.xaxis.set_major_locator(MonthLocator())
    ax_left.xaxis.set_major_formatter(DateFormatter('%b'))
    ax_left.set_xlim(year_start, year_end)   # force exact limits
    ax_left.margins(x=0)                     # remove left/right padding
    ax_left.set_xlabel("")

    # Also remove margins on twin axis to be safe
    ax_right.set_xlim(year_start, year_end)
    ax_right.margins(x=0)

    # Legend
    # --- Legend below x-axis ---
    ax_left.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.08),  # center horizontally, push below plot
        ncol=2,                       # put Obs + Sim in one row
        frameon=False,
        fontsize=10
    )

    # Make space for the legend
    plt.subplots_adjust(bottom=0.22)
    fig.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
plot_year_onepanel(2013)